# Measuring fan blade width with a laser and a photodiode

This notebook puts the ideas from `nidaqmx_tutorial.ipynb` into a small experiment.
A laser beam passes through a spinning fan onto a photodiode. Each time a blade passes,
it blocks the beam and the photodiode signal jumps up. (On this rig the photodiode reads
near 0 V while the beam is clear and high while it's blocked.) We record a short snippet of signal
around every blade arriving at the beam, plus how long each blade blocks it, and use two
Gaussian fits to measure:

* the **pulse width** τ, i.e. how long a blade takes to pass the beam
* the **blade period** T, i.e. the time between one blade and the next

From these, together with the number of blades and where the beam crosses the fan, we get the blade width.

```
          laser ───────────────► ┃ ◄─ fan blade ─► photodiode ──► AI0
                                  (beam at radius r from the hub)

 photodiode        ┌─────┐         ┌─────┐         ┌─────┐
 signal            │     │         │     │         │     │
              ─────┘     └─────────┘     └─────────┘     └────
                   ◄─ τ ─►
                   ◄────── T ──────►
```

The fan and the laser run from their **own power supplies**, at a fixed speed and
brightness. The DAQ only records: the photodiode is the only thing connected to it.

### The physics

The fan has `N` blades, so it turns once every `N·T` seconds. Its rotation frequency is

$$f = \frac{1}{N\,T}$$

At radius $r$ the blade moves at $v = 2\pi r f$, so the blade width is

$$w = v\,\tau = \frac{2\pi r\,\tau}{N\,T}$$

### Why Gaussians?

**The edge.** A laser beam's brightness falls off as a Gaussian from its centre. As a straight
blade edge sweeps across the beam, the light it blocks rises smoothly (as a cumulative
Gaussian), and the **slope** of the photodiode signal is the Gaussian itself, peaking at the
moment the edge crosses the centre of the beam. Fitting a Gaussian to the slope of the
averaged blade arrivals gives:

* its **width** σ (in time): the beam's 1/e² diameter is $d = 4\,v\,\sigma$. This is the
  *knife-edge* method used to measure laser spots on real microscopes.
* its **centre** $t_\text{edge}$, relative to the threshold crossing. Unless the threshold is
  exactly halfway up the edge, the time spent above the threshold is not quite τ: the
  threshold is crossed $t_\text{edge}$ early on the way up and $t_\text{edge}$ late on the way
  down, so $\tau = \tau_\text{threshold} - 2\,t_\text{edge}$.

**The width.** Each of the 1000 blade passes gives its own width $v\,\tau$. They vary a
little, and a histogram of them is close to a Gaussian distribution: the fitted centre is
the blade width, and its σ is how much single blade passes vary.

$w$ depends only on the **ratio** τ/T. We don't need to know the fan's speed in advance,
because T is measured in the same recording, so small changes in fan speed between
recordings shouldn't change the result. Section 4 checks this, and also checks that the
sample rate is high enough.

### How this notebook follows the exercises

This notebook is the worked version of the fan exercises. Each section matches one script,
with the same settings, function names and steps, so you can compare your code with it.
Try the exercise first.

| Section | Exercise | Tutorial idea |
|---|---|---|
| 1. Align the beam | `05_fan_live_view.py` | Continuous AI loop / live plot (4a, 4b) |
| 2. Record the blade passes | `06_fan_record.py` | Continuous AI read in chunks (4a) |
| 3. Measure the blade width | `07_fan_analyse.py` | numpy, plus `scipy.optimize.curve_fit` |
| 4. Checking the measurement | Beyond the exercises: repeatability and sample rate | Choosing `RATE` for hardware-timed AI (1b) |

### Wiring and safety

| Signal | DAQ pin | Notes |
|---|---|---|
| Photodiode | `AI0` (RSE) | Use a photodiode amplifier, or a photodiode with a load resistor, so you get a voltage. Ground goes to AI GND. |
| Fan | not connected | Runs from its own power supply at a fixed speed. |
| Laser | not connected | Runs from its own power supply. |

No analog outputs are used in this notebook. **If the AO0 → AI0 loopback wire from the
exercises is still connected, remove it.** Otherwise AI0 measures the AO output instead of
your photodiode.

⚠️ **Laser safety:** never look into the beam, and watch out for reflections off the fan blades.
Use a low-power module (class 2 or below) and keep the beam path below eye level.
⚠️ Keep fingers away from the spinning fan. It keeps spinning when your code stops,
so switch it off at its own supply before reaching near it.

---
## Why a fan? The neuroscience connection

This setup is a small, safe model of two instruments that neuroscience labs use every day.

### A scanning (multiphoton) microscope

A two-photon microscope has **no camera**. It focuses a pulsed laser to a single point, moves
that point across the tissue with scan mirrors, and records the fluorescence from each
position with a single detector, a photomultiplier tube (PMT). The image only exists
because we know **where the beam was at each moment**: time is converted into position.

Our fan setup does the same thing the other way round. The beam stays still and the object
moves through it. We record the photodiode over time and use the known speed to turn time into distance:

$$\text{size} = \text{speed} \times \text{time}$$

That is exactly how a microscope's pixel size is set: µm per pixel = scan speed × time per pixel.
It's also why scan speed has to be known precisely. A **resonant scanner** (typically ~8 kHz)
moves sinusoidally, so it travels fastest in the middle of each line and slows at the edges,
and imaging software has to correct for that or the image is distorted at the edges. Measuring the fan's
rotation from the blade period T is like measuring a resonant scanner's actual frequency
from its sync output.

### An electrophysiology rig

Look at the photodiode trace and think "electrode": a steady baseline interrupted by brief,
stereotyped events. Finding the blade pulses is the same problem as **spike detection**:

* set a threshold relative to the baseline and the noise
* find the threshold crossings
* cut out a short snippet around each crossing and overlay them, to see the event's shape
* measure each event's **width** and the **interval** between events

The photodiode itself is standard kit in **optogenetics**. A photodiode picking off some of
the stimulation light is how labs check that the light came on when it was commanded, and
for as long. Here the laser runs from its own supply, so we only use the measuring side of that.

### How the parts map

| This experiment | Two-photon microscope | Ephys / optogenetics rig |
|---|---|---|
| Laser beam | Excitation laser | Optogenetic laser/LED |
| Fan blades moving through the beam | Scan mirrors moving the beam over the tissue | — |
| Photodiode on AI0 | PMT detecting fluorescence | Electrode + amplifier, or a light-monitoring photodiode |
| Unblocked level (beam clear, near 0 V) | PMT offset, subtracted from images | Baseline |
| Blade pulse | Beam passing over a labelled cell | A spike |
| 1000 snippets overlaid and averaged | Averaging repeated frames | Spike waveforms overlaid; a spike-triggered average |
| Pulse width τ → blade width | Time per pixel → pixel size | Spike width |
| Blade period T | Line period of the scanner | Inter-spike interval → firing rate |
| Hardware sample clock: sample *i* is at *i* / `RATE` | Pixel clock: sample *i* is pixel *i* | The amplifier's fixed sampling rate |

The 🧠 notes through the notebook point out where each step shows up in a real experiment.

---
## 0. Setup

Leave `SIMULATE = True` to run the whole notebook without hardware. It generates a
realistic fake photodiode signal, so you can check the analysis first.
Set it to `False` once everything is wired up.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf

import nidaqmx
from nidaqmx.constants import AcquisitionType, TerminalConfiguration

SIMULATE = True   # <-- set False to use the real DAQ

### Configuration

Everything you may need to change is in this one cell.

In [ ]:
# --- DAQ channel ---
DEV = "Dev1"           # device name, as shown in NI MAX
PD_CHAN = f"{DEV}/ai0"

# --- Acquisition (exercises 05 and 06) ---
RATE = 20_000          # samples/s. Check your device's maximum (e.g. USB-6001 is 20 kS/s)
CHUNK = 2_000          # samples per read (10 reads per second)
THRESHOLD_V = 1.0      # a blade arriving makes the signal rise through this (V)
PRE = 40               # samples kept before each crossing } short enough that a snippet
POST = 40              # samples kept after each crossing  } holds only the rising edge
N_SNIPPETS = 1000      # blade passes to record
MIN_GAP_S = 0.5e-3     # ignore crossings this soon after the last one (noise on the same edge)
MAX_SECONDS = 30.0     # give up if there aren't enough blade passes by then
DATA_DIR = Path("..") / "data"   # recordings go here, as in exercise 06

# --- The fan (exercise 07: measure these!) ---
N_BLADES = 7
BEAM_RADIUS_MM = 30.0  # distance from the fan's axis to the laser beam
EDGE_FIT_MS = 0.5      # fit the edge's Gaussian within this time of t = 0

### Fake data for `SIMULATE = True`

This models a fan spinning at about 20 revolutions per second, with blades about 15 mm wide
(each slightly different), a Gaussian laser beam 1 mm across, a slight wobble in speed and a
little noise. Like the rig, it reads near 0 V while the beam is clear and 2 V while it's
blocked. Like a real fan on its own supply, its speed varies by a few percent from one
recording to the next. It has nothing to do with DAQmx. It's only there so the analysis can
be tested against a known answer.

In [ ]:
SIM_TRUE_BLADE_MM = 15.0
SIM_BEAM_DIAMETER_MM = 1.0  # 1/e² diameter of the Gaussian beam
SIM_FAN_HZ = 20.0           # rotation frequency (revolutions per second)
SIM_SPEED_JITTER = 0.03     # 3 % speed variation between recordings
SIM_CLEAR_V, SIM_BLOCKED_V = 0.05, 2.0   # like the rig: high while a blade blocks the beam
rng = np.random.default_rng(0)


def fake_reader(rate, chunk):
    """Returns a function that reads the next `chunk` samples of a fake fan."""
    blade_mm = SIM_TRUE_BLADE_MM + rng.normal(0, 0.15, N_BLADES)     # the blades differ slightly
    speed = 2 * np.pi * BEAM_RADIUS_MM * SIM_FAN_HZ * (1 + rng.normal(0, SIM_SPEED_JITTER))  # mm/s
    pitch = 2 * np.pi * BEAM_RADIUS_MM / N_BLADES                    # mm from one blade to the next
    start = rng.uniform(0, 100 * pitch)
    sigma = SIM_BEAM_DIAMETER_MM / 4
    n_done = 0

    def read_chunk():
        nonlocal n_done
        t = (n_done + np.arange(chunk)) / rate
        n_done += chunk
        x = start + speed * t + 0.01 * speed / (2 * np.pi) * np.sin(2 * np.pi * t)  # mm travelled
        k = np.floor((x + (pitch - SIM_TRUE_BLADE_MM) / 2) / pitch)  # nearest blade
        s = x - k * pitch                                            # beam position from its leading edge
        w = blade_mm[k.astype(int) % N_BLADES]
        blocked = (erf(s / (np.sqrt(2) * sigma)) - erf((s - w) / (np.sqrt(2) * sigma))) / 2
        volts = SIM_CLEAR_V + (SIM_BLOCKED_V - SIM_CLEAR_V) * blocked
        return volts + rng.normal(0, 0.02, chunk)

    return read_chunk

---
## 1. Align the beam (exercise 05)

Switch the laser and the fan on at their own power supplies, then watch the signal live while
you line up the laser, the fan and the photodiode. This is the continuous loop from tutorial
section 4b. Exercise 05 does the same in its own window with pyqtgraph, which redraws much
faster than a notebook can.

* With the fan spinning, you should see a regular train of pulses. The bottom of the trace is the
  **unblocked** level, the top of each pulse the **blocked** level when a blade fully blocks the beam.
* Adjust the alignment until the contrast (blocked − unblocked) in the title is as large as you can
  get it. Below about 0.1 V is too small to analyse.
* Note the two levels: `THRESHOLD_V` in section 2 goes roughly halfway between them.

In [ ]:
from IPython.display import clear_output, display


def live_view(seconds=10.0, chunk=5_000, y_max=6.0):
    """Plot the photodiode signal live, with its two levels (as exercise 05)."""
    if SIMULATE:
        print("[sim] live view needs hardware; skipping")
        return
    fig, ax = plt.subplots(figsize=(10, 3))
    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        task.timing.cfg_samp_clk_timing(
            rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * chunk
        )
        task.start()
        t_end = time.time() + seconds
        while time.time() < t_end:
            data = np.array(task.read(number_of_samples_per_channel=chunk))
            low, high = np.percentile(data, [5, 95])
            ax.clear()
            ax.plot(np.arange(chunk) / RATE * 1000, data, lw=0.8)
            ax.set_ylim(-0.2, y_max)
            ax.set_xlabel("Time within update (ms)")
            ax.set_ylabel("Photodiode (V)")
            ax.set_title(f"unblocked ≈ {low:.2f} V    blocked ≈ {high:.2f} V    contrast {high - low:.2f} V")
            clear_output(wait=True)
            display(fig)
    plt.close(fig)


live_view(seconds=10)

---
## 2. Record the blade passes (exercise 06)

The DAQ streams continuously and we read it in chunks, as in tutorial section 4a. After each
chunk we look for places where the signal **rises through** `THRESHOLD_V`, i.e. a blade
arriving at the beam. For each one we keep `PRE` samples before the crossing and `POST`
samples after it, so every snippet looks like

    0, 0, 0, 0, ... gradual rise ..., top, top, top, ...

We also note how many samples each blade keeps the signal above the threshold, by finding
where it **drops back** through it. That's everything exercise 07 needs. `check_window`
stops with suggested values if `PRE` reaches back to the previous blade, or `POST` runs on
until this one leaves.

Set `THRESHOLD_V` roughly halfway between the two levels you saw in the live view. Its exact
value doesn't matter for the result: section 3 corrects for where it sits on the edge.

The DAQ's own clock times every sample, so sample *i* was taken at exactly *i* / `RATE`
seconds. That's what lets us turn sample counts into times, and line snippets up to within
a sample.

> **🧠 In the lab:** this is exactly how many ephys acquisition systems store spikes. They
> stream continuously, detect threshold crossings, and save a short snippet (typically 1–2 ms)
> around each one. Overlaying the snippets shows the waveform's shape, and averaging them
> (a *spike-triggered average*) pulls a clean shape out of noisy single events. The noise
> falls by √N: by about 30× for 1000 snippets.

In [ ]:
def find_crossings(signal, threshold):
    """Sample indices where `signal` rises through `threshold` (a blade arriving)."""
    return np.flatnonzero((signal[:-1] < threshold) & (signal[1:] >= threshold)) + 1


def find_falls(signal, threshold):
    """Sample indices where `signal` drops back through `threshold` (a blade leaving)."""
    return find_crossings(-signal, -threshold)


def check_window(rises, blocked, pre=PRE, post=POST):
    """Stop if pre and post would show more than the rising edge."""
    if len(blocked) < 10:
        return
    period = np.mean(np.diff(rises))  # samples from one blade to the next
    on = np.median(blocked)           # samples the beam stays blocked
    off = period - on                 # samples the beam is clear between blades
    if pre > 0.8 * off or post > 0.8 * on:
        raise RuntimeError(
            f"The beam is clear for about {off:.0f} samples, then blocked for about {on:.0f}. "
            f"With PRE = {pre} and POST = {post} a snippet would reach the previous blade or "
            f"this blade leaving. Try PRE = {0.6 * off:.0f} and POST = {0.6 * on:.0f}.")


def collect(read_chunk, rate, pre=PRE, post=POST):
    """Read chunks until there are N_SNIPPETS complete snippets.

    Returns (snippets, rises, blocked, data): one row of snippets per blade pass, the
    sample index in data where each blade arrived, and how many samples each one kept
    the signal above THRESHOLD_V.
    """
    data = np.empty(0)
    rises = np.empty(0, dtype=int)
    gap = MIN_GAP_S * rate
    t_end = time.time() + MAX_SECONDS
    while time.time() < t_end:
        data = np.concatenate([data, read_chunk()])
        rises = find_crossings(data, THRESHOLD_V)
        falls = find_falls(data, THRESHOLD_V)
        # Like a refractory period in spike detection: only count a rise if the
        # signal has been below the threshold for at least MIN_GAP_S.
        prev_fall = np.searchsorted(falls, rises) - 1
        rises = rises[(prev_fall < 0) | (rises - falls[np.maximum(prev_fall, 0)] > gap)]
        # The first drop at least MIN_GAP_S after each rise is that blade leaving.
        nxt = np.searchsorted(falls, rises + gap)
        rises = rises[nxt < falls.size]
        blocked = falls[nxt[nxt < falls.size]] - rises
        check_window(rises, blocked, pre, post)
        # Keep only rises with pre samples before and post samples after.
        keep = (rises >= pre) & (rises + post <= data.size)
        rises, blocked = rises[keep], blocked[keep]
        if len(rises) >= N_SNIPPETS:
            rises, blocked = rises[:N_SNIPPETS], blocked[:N_SNIPPETS]
            snippets = np.stack([data[r - pre:r + post] for r in rises])
            return snippets, rises, blocked, data

    low, high = np.percentile(data, [5, 95])
    raise RuntimeError(f"Only {len(rises)} blade passes in {MAX_SECONDS:.0f} s. Set THRESHOLD_V "
                       f"between the clear ({low:.2f} V) and blocked ({high:.2f} V) levels.")


def record(rate=RATE, pre=PRE, post=POST):
    """Record N_SNIPPETS blade passes. Returns (snippets, rises, blocked, data)."""
    if SIMULATE:
        return collect(fake_reader(rate, CHUNK), rate, pre, post)

    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        task.timing.cfg_samp_clk_timing(
            rate=rate, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * CHUNK
        )
        task.start()

        def read_chunk():
            return np.asarray(task.read(number_of_samples_per_channel=CHUNK))

        return collect(read_chunk, rate, pre, post)

In [ ]:
snippets, rises, blocked, data = record()

DATA_DIR.mkdir(exist_ok=True)
prefix = "sim_fan_snippets" if SIMULATE else "fan_snippets"
path = DATA_DIR / f"{prefix}_{time.strftime('%Y%m%d_%H%M%S')}_{RATE}Hz.npz"
np.savez(path, snippets=snippets, rises=rises, blocked=blocked, rate=RATE, pre=PRE,
         threshold=THRESHOLD_V)

ms = blocked / RATE * 1000
print(f"Each blade blocked the beam for {ms.mean():.2f} ms on average ({ms.min():.2f} to {ms.max():.2f} ms)")
low, high = np.percentile(data, [5, 95])
print(f"Saved {path.name}  clear {low:.2f} V, blocked {high:.2f} V")

t_ms = np.arange(-PRE, POST) / RATE * 1000
plt.figure(figsize=(10, 4))
plt.plot(t_ms, snippets.T, color="tab:blue", lw=0.5, alpha=0.05)
plt.plot(t_ms, snippets.mean(axis=0), color="k", lw=1.5, label="mean")
plt.axhline(THRESHOLD_V, color="gray", ls=":", label="threshold")
plt.xlabel("Time from threshold crossing (ms)")
plt.ylabel("Photodiode (V)")
plt.title(f"{len(snippets)} blade passes, overlaid")
plt.legend(loc="upper left")
plt.show()

### Reading the overlay (exercise 06's questions)

* Each edge takes only a few samples to rise. That's set by the beam's width and the blade's
  speed (about 1 mm at nearly 4 m/s, i.e. about 0.25 ms), not by the sample rate.
* Moving `THRESHOLD_V` slides every trace sideways so that its crossing is still at t = 0,
  but it doesn't change their shape. It *does* change the time blocked: a higher threshold
  is crossed later on the way up and earlier on the way down. Section 3 corrects for this.
* The time blocked varies from pass to pass because the fan's speed wobbles slightly, the
  blades aren't all identical, and each crossing can only be placed to the nearest sample.

> **🧠 In the lab:** spike waveforms behave the same way. Overlaid on their threshold
> crossing, they're tightly aligned near the crossing and spread out further away, which is
> why spike-sorting and averaging windows are kept short.

---
## 3. Measure the blade width (exercise 07)

The steps below follow exercise 07's TODOs in order. Exercise 07 loads the recording from the
file saved above. Here it's already in memory.

### The edge: mean and slope (TODO 1)

Averaging 1000 snippets lowers the noise by about √1000 ≈ 30×. The slope of the mean is the
difference between neighbouring samples times `RATE` (V/s). It belongs halfway between the two
samples, hence `t_mid`.

Sample `PRE` in each snippet is the first at or above the threshold, so the signal actually
crossed it somewhere in the sample before: on average half a sample earlier. We call that
moment t = 0. (Without the half sample, the width comes out one sample's worth too wide.)

In [ ]:
rate, pre = RATE, PRE
t = (np.arange(snippets.shape[1]) - pre + 0.5) / rate   # s, 0 = threshold crossing
t_mid = t[:-1] + 0.5 / rate                             # times between samples, for np.diff

mean = snippets.mean(axis=0)
slope = np.diff(mean) * rate

### A Gaussian and how to fit it (TODOs 2–3)

`curve_fit` adjusts the Gaussian's height, centre and width until it matches the data as
closely as possible (least squares), starting from a rough guess. We use the same
`fit_gaussian` twice: for the edge here, and for the histogram of widths below.

In [ ]:
def gaussian(x, amplitude, centre, sigma):
    """A Gaussian of height `amplitude`, centred on `centre`, with width `sigma`."""
    return amplitude * np.exp(-(x - centre) ** 2 / (2 * sigma ** 2))


def fit_gaussian(x, y, guess):
    """Fit `gaussian` to the points (x, y), starting from `guess`. Returns (amplitude, centre, sigma)."""
    params, _ = curve_fit(gaussian, x, y, p0=guess)
    amplitude, centre, sigma = params
    return amplitude, centre, abs(sigma)


# Fit the slope near t = 0, starting from a peak as high as the biggest slope, at t = 0, about 2 samples wide.
near = np.abs(t_mid) <= max(EDGE_FIT_MS / 1000, 4 / rate)
edge_fit = fit_gaussian(t_mid[near], slope[near], guess=[slope.max(), 0.0, 2 / rate])
_, t_edge, sigma_edge = edge_fit

### The blade's speed (TODO 4)

* **Blade period** T: the mean time between successive blades arriving.
* **Rotation frequency** f = 1 / (N·T), **blade speed at the beam** v = 2πrf.

In [ ]:
blade_period = np.mean(np.diff(rises)) / rate
rot_freq = 1 / (N_BLADES * blade_period)
speed_mm_s = 2 * np.pi * BEAM_RADIUS_MM * rot_freq

### The width: a histogram and a Gaussian (TODOs 5–6)

* **Beam diameter** d = 4vσ, from the edge's Gaussian.
* **Width of each blade pass** w = v × (time blocked − 2·t_edge).

Each time blocked is a whole number of samples, so the widths come in steps of one sample's
worth of blade travel (v / `RATE`, about 0.19 mm here). The histogram uses one bar per step,
and a Gaussian fitted to it gives the blade width (its centre) and the spread of single blade
passes (its σ).

In [ ]:
beam_mm = 4 * speed_mm_s * sigma_edge
widths_mm = speed_mm_s * (blocked / rate - 2 * t_edge)

step = speed_mm_s / rate
bin_edges = np.arange(widths_mm.min() - step / 2, widths_mm.max() + step, step)
counts, bin_edges = np.histogram(widths_mm, bins=bin_edges)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2

hist_fit = fit_gaussian(bin_centres, counts, guess=[counts.max(), widths_mm.mean(), widths_mm.std()])
_, w_mm, w_sigma = hist_fit

print(f"Fan speed:     {rot_freq * 60:.0f} rpm (blade period {blade_period * 1e3:.2f} ms)")
print(f"Edge:          centre {t_edge * 1e3:+.3f} ms from the threshold crossing, σ {sigma_edge * 1e3:.3f} ms")
print(f"Beam diameter: {beam_mm:.2f} mm (1/e²)")
print(f"Blade width:   {w_mm:.2f} mm (σ {w_sigma:.2f} mm over {len(widths_mm)} blade passes)")
if SIMULATE:
    print(f"[sim] true blade width about {SIM_TRUE_BLADE_MM} mm, beam {SIM_BEAM_DIAMETER_MM} mm")

In [ ]:
t_ms = t * 1e3
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9, 9))
ax1.plot(t_ms, snippets[:200].T, color="tab:blue", lw=0.5, alpha=0.05)
ax1.plot(t_ms, mean, color="k", lw=1.5, label="mean")
ax1.axvline(t_edge * 1e3, color="tab:red", ls="--", label="fitted edge centre")
ax1.set_xlabel("Time from threshold crossing (ms)")
ax1.set_ylabel("Photodiode (V)")
ax1.set_title("Blade arriving")
ax1.legend(loc="upper left")

ax2.plot(t_mid * 1e3, slope / 1e3, color="k", lw=1, label="slope of the mean")
ax2.plot(t_mid * 1e3, gaussian(t_mid, *edge_fit) / 1e3, color="tab:red", ls="--",
         label=f"Gaussian: beam {beam_mm:.2f} mm")
ax2.set_xlabel("Time from threshold crossing (ms)")
ax2.set_ylabel("Slope (V/ms)")
ax2.legend(loc="upper left")

x = np.linspace(bin_edges[0], bin_edges[-1], 300)
ax3.bar(bin_centres, counts, width=np.diff(bin_edges), color="tab:blue", alpha=0.5, label="blade passes")
ax3.plot(x, gaussian(x, *hist_fit), color="tab:red", ls="--", label=f"Gaussian: {w_mm:.2f} ± {w_sigma:.2f} mm")
ax3.set_xlabel("Blade width (mm)")
ax3.set_ylabel("Blade passes")
ax3.legend(loc="upper left")
plt.tight_layout()
plt.show()

> **🧠 In the lab:** `speed × time = distance` is the whole calculation behind a scanning
> microscope's pixel size. For example, a resonant scanner at 8 kHz that sweeps 500 µm in roughly
> half of each 125 µs period, sampled into 512 pixels, gives about 1 µm per pixel and about 0.1 µs per
> pixel. In ephys, the same per-event numbers become **spike width** and **firing rate**
> (1 / inter-spike interval). Spike width is often used to tell narrow-spiking, putative
> fast-spiking interneurons apart from broad-spiking, putative pyramidal cells.

The fitted σ is the same measurement as a **knife-edge** scan of a microscope's laser focus:
move a sharp edge across the beam, and the slope of the transmitted light is the beam's profile.

### Exercise 07's questions

* **Ruler:** fan blades are angled (pitched), so the beam measures the blade's width along the
  direction of rotation at radius r, not along its surface. Expect the ruler to read more.
* **Changing the threshold** changes the average time blocked, but the edge's centre moves
  by the same amount relative to the crossing, so the − 2·t_edge correction cancels it and the
  width stays the same.
* **The histogram's σ** comes from real pass-to-pass variation (speed wobble, blades that
  differ) and from placing each crossing to the nearest sample. Recording more passes doesn't
  shrink it, but it does shrink the uncertainty of the fitted centre, roughly as σ / √N.
* **Precision beyond one sample:** each crossing lands at a random point within a sample, so
  the rounding averages out over 1000 passes. At a much lower rate the steps get coarser,
  the histogram collapses to one or two bars, and the Gaussian fit stops working.

### The same steps as one function

The per-blade comparison and section 4 repeat the analysis many times, so here it is in one function.

In [ ]:
def analyse(snippets, rises, blocked, rate, pre):
    """Exercise 07's steps in one go. Returns (widths_mm, width_mm, beam_mm, rot_freq)."""
    t = (np.arange(snippets.shape[1]) - pre + 0.5) / rate
    t_mid = t[:-1] + 0.5 / rate
    slope = np.diff(snippets.mean(axis=0)) * rate
    near = np.abs(t_mid) <= max(EDGE_FIT_MS / 1000, 4 / rate)
    _, t_edge, sigma_edge = fit_gaussian(t_mid[near], slope[near], guess=[slope.max(), 0.0, 2 / rate])

    rot_freq = 1 / (N_BLADES * np.mean(np.diff(rises)) / rate)
    speed_mm_s = 2 * np.pi * BEAM_RADIUS_MM * rot_freq
    widths_mm = speed_mm_s * (blocked / rate - 2 * t_edge)

    step = speed_mm_s / rate
    edges = np.arange(widths_mm.min() - step / 2, widths_mm.max() + step, step)
    counts, edges = np.histogram(widths_mm, bins=edges)
    centres = (edges[:-1] + edges[1:]) / 2
    if len(counts) < 4:
        raise RuntimeError(f"widths fall into only {len(counts)} histogram bars, too few to fit")
    _, width_mm, _ = fit_gaussian(centres, counts, guess=[counts.max(), widths_mm.mean(), widths_mm.std()])
    return widths_mm, width_mm, 4 * speed_mm_s * sigma_edge, rot_freq


print(f"Check: {analyse(snippets, rises, blocked, RATE, PRE)[1]:.2f} mm, as above")

### Each blade separately

Blade passes arrive in a fixed order, so counting blade periods from the first one tells us
which blade made each pass. Averaging each blade's widths shows whether the blades differ.
Which group is "blade 1" is arbitrary, because we don't know which blade passed first.

In [ ]:
blade_index = np.round((rises - rises[0]) / (blade_period * RATE)).astype(int) % N_BLADES
per_blade = [widths_mm[blade_index == b].mean() for b in range(N_BLADES)]

plt.figure(figsize=(6, 3))
plt.plot(range(1, N_BLADES + 1), per_blade, "o")
plt.axhline(w_mm, color="k", ls=":", label="all blades")
plt.xlabel("Blade (arbitrary numbering)")
plt.ylabel("Width (mm)")
plt.title("Width of each blade")
plt.legend()
plt.show()

print("Widths:", np.round(per_blade, 2), "mm")

If one blade is clearly offset from the others, check that `N_BLADES` is correct.
If it's wrong, the groups mix different blades together and the rotation frequency is off by a factor.

> **🧠 In the lab:** sorting events by "which blade made them" is a very simple version of
> **spike sorting**, i.e. working out which neuron each spike on an electrode came from. Here it's
> easy because the blades arrive in a fixed order. With real neurons you group spikes by waveform
> shape instead, and getting the number of groups wrong causes the same kind of mixing.

---
## 4. Checking the measurement

We can't change the fan's speed, but we can still test the measurement in two ways.

### 4a. Is it repeatable?

Record several times in a row. A fan on its own supply doesn't hold a perfectly constant
speed: it drifts a little with warm-up, supply voltage and air flow, so the rpm will differ
slightly between recordings. Because the width depends only on τ/T, it **shouldn't** follow
those changes. If the width moves with the rpm, the edges are being measured wrongly.

In [ ]:
N_REPEATS = 5
repeats = []

for i in range(N_REPEATS):
    snip_i, rises_i, blocked_i, _ = record()
    _, w_i, beam_i, f_i = analyse(snip_i, rises_i, blocked_i, RATE, PRE)
    repeats.append((f_i * 60, w_i, beam_i))
    print(f"Recording {i + 1}: {f_i * 60:6.0f} rpm, width {w_i:6.2f} mm, beam {beam_i:.2f} mm")

rpm, width, beam = np.array(repeats).T
print(f"\nBlade width over {N_REPEATS} recordings: {width.mean():.2f} ± {width.std():.2f} mm")

### 4b. Is the sample rate high enough?

Each edge only lasts a few samples, and each time blocked is a whole number of samples. The
averaging over 1000 passes places things to a fraction of a sample, but only while the edge
spans a few samples. Record at several rates, keeping the snippet the same length in time:
the width should level off as the rate goes up, while the beam diameter is overestimated at
low rates, where the edge is narrower than a sample. Keep every rate at or below your
device's maximum.

> **🧠 In the lab:** this is the same choice as an ephys sampling rate. A spike is about 1 ms
> long, so you need ≥20 kHz to see its shape and measure its width, even though a few kHz
> would be enough to just count spikes. In a microscope, the time per pixel sets how many PMT
> samples go into each pixel. A detector or amplifier that is too slow (low bandwidth) has a
> similar effect: it rounds off the blade edges, and the fitted beam looks bigger than it is.

In [ ]:
SAMPLE_RATES = [2_000, 5_000, 10_000, 20_000]   # don't go above your device's maximum
by_rate = []

for r in SAMPLE_RATES:
    pre_r, post_r = PRE * r // RATE, POST * r // RATE   # same snippet length in time
    snip_r, rises_r, blocked_r, _ = record(rate=r, pre=pre_r, post=post_r)
    try:
        _, w_r, beam_r, _ = analyse(snip_r, rises_r, blocked_r, r, pre_r)
        by_rate.append((w_r, beam_r))
        print(f"{r:6d} samples/s: width {w_r:6.2f} mm, beam {beam_r:.2f} mm")
    except RuntimeError as err:   # too few histogram bars, or curve_fit can't converge
        by_rate.append((np.nan, np.nan))
        print(f"{r:6d} samples/s: fit failed ({err})")

rate_width, rate_beam = np.array(by_rate).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(rpm, width, "o")
ax1.axhline(width.mean(), color="k", ls=":")
ax1.set_xlabel("Speed (rpm)")
ax1.set_ylabel("Blade width (mm)")
ax1.set_title("4a: repeated recordings (should be flat)")

ax2.plot(SAMPLE_RATES, rate_width, "o-", label="blade width")
ax2.set_xscale("log")
ax2.set_xlabel("Sample rate (samples/s)")
ax2.set_ylabel("Blade width (mm)")
ax2b = ax2.twinx()
ax2b.plot(SAMPLE_RATES, rate_beam, "s--", color="tab:orange", label="beam diameter")
ax2b.set_ylabel("Beam diameter (mm)", color="tab:orange")
ax2.set_title("4b: results vs sample rate")
plt.tight_layout()
plt.show()

---
## 5. Finishing up

The DAQ doesn't output anything in this notebook, so there is nothing to reset in software.
Switch the laser and the fan off **at their own power supplies**, and wait for the fan to
stop before touching the setup.

## Things to try

* **Measure with a ruler** and compare. Fan blades are angled (pitched), so what the beam measures
  is the blade's width along the direction of rotation at radius r, not along its surface.
* **Move the beam to a different radius.** Many blades get wider towards the tip.
* **Record the leaving edge too:** also keep snippets around each drop, fit its Gaussian, and
  compare its σ with the arriving edge's. Is the leaving edge blurrier? Why might it be?
* **Fit the signal instead of its slope:** fit a cumulative Gaussian (`scipy.special.erf`) to the
  mean snippet directly. Do you get the same centre and σ?
* **How steady is the fan?** `np.diff(rises) / RATE` is the time between every pair of blades.
  Plot it against time. This is how you would check a resonant scanner's frequency for drift.
* **Check the beam diameter** by moving a razor blade across the beam on a translation stage
  while watching `live_view()`, and compare it with the fitted value.
* **Make an image:** record for one full rotation and reshape the signal into rows, one per
  rotation, over several rotations. You've built a (very slow) scanning imager. Compare it with
  how a microscope reshapes the PMT signal into a frame.